# KORE Colab GPU 실행

무료 Colab의 T4 GPU로 생성 모델을 올려서 점검하고, 개발용 실행, 문서 기여도 라벨링, 블라인드 채점표 만들기까지 한다.
셀은 위에서 아래로 한 번씩 실행한다. 각 셀은 혼자서 돌아가도록 만들어서 복사해 붙여도 된다.

| 순서 | 셀 | 하는 일 | 걸리는 시간(어림) |
|---|---|---|---|
| 1 | 준비 | 저장소 받기, 설치, 테스트, 환경 점검 | 3~5분 |
| 2 | GPU 점검 | 모델 로딩, 한국어 독해, 로그확률 계산, 시간 예측 | 5~10분(모델 내려받기 포함) |
| 3 | 개발용 실행 | no_retrieval / vanilla_rag / kore / gold_context 생성과 점검 | 질문 수와 GPU에 따라 다르다 |
| 4 | 기여도 라벨링 | 부분집합별 정답 로그확률, H1·H2 예비 지표 | 2번 셀에서 잰 호출 시간으로 어림한다 |
| 5 | 블라인드 채점표 | 사람이 조건을 모른 채 채점할 표 만들기 | 1분 |

지켜야 할 것
- 무료 Colab은 GPU 배정과 사용 시간이 보장되지 않는다. 결과는 Drive에 남기고, 끊기면 같은 셀을 다시 실행한다(끝난 부분은 건너뛴다).
- 비공개 저장소는 토큰을 노트북에 붙여 넣지 않는다. 왼쪽 열쇠 아이콘(Secrets)에 `GITHUB_TOKEN`(해당 저장소 읽기 전용)으로 저장하고 이 노트북의 접근을 켠다.
- 사용 제한을 피하려고 계정을 여러 개 돌리지 않는다. 세 사람이 각자 자기 계정으로 각자의 일을 한다.
- Colab 무료 런타임에서 웹 UI(Gradio 등)를 오래 띄우지 않는다. 데모는 A의 노트북에서 돌린다.
- `--final-test`는 이 노트북에서 켜지 않는다. 최종 평가는 사전 등록 뒤에 사람이 직접 한다.

## 1. 준비

In [ ]:
# 1. 준비: 저장소 받기, 설치, 테스트, 환경 점검
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

if 'OWNER' in REPO_URL:
    raise SystemExit('REPO_URL을 팀 저장소 주소로 바꾸거나 Colab Secrets에 KORE_REPO_URL을 저장한 뒤 다시 실행한다.')

token = _token()
url = REPO_URL.replace('https://', 'https://x-access-token:%s@' % token) if token else REPO_URL
if os.path.isdir(WORKDIR):
    sh('git fetch origin && git checkout ' + BRANCH + ' && git pull --ff-only', cwd=WORKDIR, secret=token)
else:
    sh('git clone --branch ' + BRANCH + ' ' + url + ' ' + WORKDIR, secret=token)

# Colab에 이미 있는 torch는 건드리지 않는다. transformers는 Qwen3를 읽을 수 있는 버전(4.51 이상)이면 된다.
sh(sys.executable + ' -m pip install -q -e . "transformers>=4.51" accelerate', cwd=WORKDIR)
sh(sys.executable + ' -m pytest -q', cwd=WORKDIR)
sh(sys.executable + ' -m kore doctor', cwd=WORKDIR)
print('준비 끝. 위 doctor 결과에서 GPU가 [OK]가 아니면 메뉴의 런타임 > 런타임 유형 변경에서 T4 GPU를 고른다.')

## 2. GPU 점검
모델 이름이 `configs/generator.json`과 같은지, 판정이 "통과"인지 본다. 문제가 있으면 `CONFIG`를 대체 모델 설정으로 바꿔 다시 실행한다.

In [ ]:
# GPU 점검: 모델을 올리고, 한국어 출력과 로그확률 계산이 정상인지, 시간이 얼마나 걸리는지 잰다
import math, os, sys, time
os.chdir('/content/kore')
sys.path.insert(0, '/content/kore')

CONFIG = 'configs/generator.json'  # 문제가 있으면 configs/generator_fallback_qwen25_3b.json 으로 바꿔 다시 실행한다

import torch
from kore.generation import GeneratorSpec, HFGenerator
from kore.schemas import KnowledgeItem

if not torch.cuda.is_available():
    print('경고: GPU가 없다. 런타임 > 런타임 유형 변경에서 T4 GPU를 고른다. CPU로는 이 점검이 매우 느리다.')

spec = GeneratorSpec.from_json(CONFIG)
t0 = time.time()
gen = HFGenerator(spec)
print('모델 로딩: %.0f초' % (time.time() - t0), '|', gen.name)
if torch.cuda.is_available():
    print('GPU 메모리 사용: %.1fGB / %.1fGB' % (torch.cuda.memory_allocated() / 1e9, torch.cuda.get_device_properties(0).total_memory / 1e9))

def doc(i, text):
    return KnowledgeItem(id='smoke_%d' % i, domain='smoke', title='t', text=text, source_url='local:smoke', license='own')

# 지어낸 사실이라서 모델이 미리 알 수 없다. 근거를 읽고 답하는지(한국어 독해)와 모르는 것을 지어내지 않는지 본다.
ev1 = [doc(1, '모래별섬은 면적이 12.4제곱킬로미터이고 주민은 340명이다. 이 섬의 대표 특산물은 말린 해초이다.')]
ev2 = [doc(2, '푸른여울 도서관은 1998년에 문을 열었고, 장서는 약 8만 권이다. 매주 월요일은 쉰다.')]
ev3 = [doc(3, '마루 협약에서 회원국은 매년 3월에 총회를 연다. 의결에는 회원국의 3분의 2 이상 찬성이 필요하다.')]
cases = [
    ('모래별섬의 주민은 몇 명인가?', ev1, '340'),
    ('푸른여울 도서관은 무슨 요일에 쉬는가?', ev2, '월요일'),
    ('마루 협약에서 의결에 필요한 찬성 비율은?', ev3, '3분의 2'),
    ('모래별섬의 주민은 몇 명인가?', [], None),
]
ok = 0
times = []
for q, ev, expect in cases:
    t = time.time()
    ans = gen.generate(q, ev)
    times.append(time.time() - t)
    hit = (expect in ans) if expect else None
    ok += 1 if hit else 0
    print('Q:', q, '| 근거 %d개' % len(ev), '|', '%.1f초' % times[-1])
    print('A:', ans, '|', '정답 포함' if hit else ('정답 없음' if expect else '(근거 없음: 숫자를 지어내면 안 된다)'))
print('독해 정답 %d/3, 생성 평균 %.1f초' % (ok, sum(times) / len(times)))

# 로그확률: 빠른 계산이 전체 계산과 같은 값을 내는지, 유한한 값인지, 얼마나 빠른지
answer_text = '340명이다.'
diff = gen.check_scoring(cases[0][0], ev1, answer_text)
s_with = gen.score(cases[0][0], ev1, answer_text)
s_without = gen.score(cases[0][0], [], answer_text)
print('계산 방식 차이(평균 로그확률): %.5f  (0.05 이하여야 한다)' % diff)
print('근거 있을 때 %.3f, 없을 때 %.3f  → 근거가 있으면 더 커야 한다' % (s_with, s_without))
long_ev = [doc(i, '긴 근거 문장이다. ' * 60) for i in range(3)]
t = time.time()
for _ in range(5):
    gen.score('질문?', long_ev, answer_text)
sec = (time.time() - t) / 5
print('긴 근거(3개) 점수 계산: 호출당 %.2f초' % sec)
for cands, sub in ((6, 2), (8, 3)):
    from kore.labeling import count_calls
    n = count_calls(cands, sub)
    print('후보 %d개·크기 %d까지: 질문당 %d번 ≈ %.0f초, 24문항이면 ≈ %.0f분' % (cands, sub, n, n * sec, 24 * n * sec / 60))

problems = []
if ok < 2: problems.append('근거를 읽고 답하는 한국어 독해가 약하다 → 대체 모델로 바꿔 본다')
if diff > 0.05: problems.append('로그확률 계산 방식이 어긋난다 → 이 모델 구조에서는 라벨링을 쓰지 않는다. A에게 알린다')
if not (math.isfinite(s_with) and math.isfinite(s_without)): problems.append('점수가 유한하지 않다(fp16 오버플로 의심) → 대체 모델로 바꾼다')
if s_with <= s_without: problems.append('근거를 줘도 점수가 오르지 않는다 → 프롬프트나 모델을 의심한다')
print()
print('판정:', '통과' if not problems else '문제 있음')
for p in problems: print(' -', p)
print('이 결과(모델 이름, 시간, 판정)를 그대로 복사해서 팀에 공유한다.')

## 2-1. 검색기 비교 (BM25 vs 하이브리드)
정답 근거가 후보 10개 안에 들어오는지부터 본다. 임베딩 모델(약 0.1B)은 CPU에서도 돈다. train·dev만 쓴다.

In [ ]:
# 2-1. 검색기 비교
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

KNOWLEDGE = 'data/knowledge.csv'
QUESTIONS = 'data/evaluation.csv'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
if not os.path.exists(KNOWLEDGE) or not os.path.exists(QUESTIONS):
    print('실제 데이터가 아직 없어서 샘플로 대신한다. 이 결과는 배관 점검용이다(샘플은 영어라 한국어 효과를 보여 주지 못한다).')
    KNOWLEDGE, QUESTIONS = 'data/sample_knowledge.csv', 'data/sample_eval.csv'

# (1) 임베딩 모델이 한국어 바꿔 말하기를 알아보는지 먼저 본다
from kore.dense import HFEncoder, DEFAULT_DENSE_MODEL
t0 = time.time()
enc = HFEncoder(DEFAULT_DENSE_MODEL)
print('임베딩 모델 로딩: %.0f초' % (time.time() - t0), enc.name, enc.device)
q = enc.encode(['월급은 언제 받나요?'], 'query')[0]
docs = enc.encode(['사용자는 근로자에게 매달 정해진 날에 임금을 지급해야 한다.', '연차 유급휴가는 1년에 15일을 준다.'], 'passage')
dot = lambda a, b: sum(x * y for x, y in zip(a, b))
s_pay, s_leave = dot(q, docs[0]), dot(q, docs[1])
print('월급 질문 ↔ 임금 문서 %.3f, ↔ 휴가 문서 %.3f' % (s_pay, s_leave), '→', '정상' if s_pay > s_leave else '이상함(A에게 알린다)')

# (2) 우리 데이터의 train·dev에서 검색기 비교. 첫 번째가 기준이다. test는 쓰지 않는다.
sh(sys.executable + ' -m kore retrieval-eval --knowledge ' + KNOWLEDGE + ' --questions ' + QUESTIONS +
   ' --compare bm25:ngram_stem hybrid:ngram_stem:0.5 hybrid:ngram_stem:0.3 --out results/retrieval_eval.json', stream=True)
print('하이브리드의 "후보" 차이 구간이 0보다 확실히 크면(→ 나음) 3번 셀의 PIPE_CONFIG를 configs/hybrid.json으로 바꾼다.')
print('"구분 못 함"이면 기본값(bm25)을 그대로 둔다. 바꿨다면 그 사실과 이 표를 보고서에 적는다.')

## 3. 개발용 실행
train·dev만 돈다(test는 돌지 않는다). 끝나면 `screen`이 분야 게이트(PASS/FAIL)를 보여 준다.

In [ ]:
# 3. 개발용 실행과 닫힌북 필터
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

USE_DRIVE = True
CONFIG = 'configs/generator.json'
KNOWLEDGE = 'data/knowledge.csv'      # 아직 없으면 샘플로 대신한다(결과는 연구 결과가 아니다)
QUESTIONS = 'data/evaluation.csv'
PIPE_CONFIG = 'configs/default.json'  # 2-1 검색기 비교에서 하이브리드가 확실히 나으면 configs/hybrid.json
RUN_NAME = 'dev_run_01'               # 설정이나 모델을 바꾸면 이름도 바꾼다(섞이면 오류가 난다)

os.chdir(WORKDIR)
if not os.path.exists(KNOWLEDGE) or not os.path.exists(QUESTIONS):
    print('실제 데이터가 아직 없어서 샘플로 대신한다. 이 결과는 배관 점검용이다.')
    KNOWLEDGE, QUESTIONS = 'data/sample_knowledge.csv', 'data/sample_eval.csv'

OUT_ROOT = '/content/results'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/kore_results'
OUT_DIR = OUT_ROOT + '/' + RUN_NAME

py = sys.executable
sh(py + ' -m kore validate --knowledge ' + KNOWLEDGE + ' --questions ' + QUESTIONS)
# test 분할은 돌지 않는다(--final-test를 주지 않는다). 연결이 끊기면 이 셀을 다시 실행하면 이어서 돈다.
sh(py + ' -m kore run --config ' + PIPE_CONFIG + ' --knowledge ' + KNOWLEDGE + ' --questions ' + QUESTIONS +
   ' --out ' + OUT_DIR + ' --generator-config ' + CONFIG +
   ' --conditions no_retrieval vanilla_rag kore gold_context', stream=True)
sh(py + ' -m kore screen --run-dir ' + OUT_DIR)
print('결과 폴더:', OUT_DIR)
print('screen.csv에 표시된 질문과 분야 게이트 판정을 C와 B에게 공유한다.')

In [ ]:
# 결과 표 보기(조건별 평균과 95% 구간, 조건 간 쌍체 차이)
import pandas as pd
RUN_DIR = '/content/drive/MyDrive/kore_results/dev_run_01'  # 위에서 쓴 RUN_NAME에 맞춘다
display(pd.read_csv(RUN_DIR + '/results.csv'))
display(pd.read_csv(RUN_DIR + '/comparisons.csv'))

## 4. 기여도 라벨링 (H1·H2 예비 신호)
먼저 dev만, 후보 6개·크기 2까지(질문당 22번)로 돌린다. 신호가 있을 때만 넓힌다.

In [ ]:
# 4. 문서 기여도 라벨링
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

USE_DRIVE = True
CONFIG = 'configs/generator.json'
KNOWLEDGE = 'data/knowledge.csv'
QUESTIONS = 'data/evaluation.csv'
LABEL_NAME = 'dq_light_01'
SPLITS = 'dev'          # 먼저 dev만. H1 신호가 있으면 'train dev'로 넓힌다
MAX_CANDS, MAX_SUBSET = 6, 2   # 질문당 22번. 시간이 남으면 8, 3(93번)

os.chdir(WORKDIR)
if not os.path.exists(KNOWLEDGE) or not os.path.exists(QUESTIONS):
    print('실제 데이터가 아직 없어서 샘플로 대신한다. 이 결과는 배관 점검용이다.')
    KNOWLEDGE, QUESTIONS = 'data/sample_knowledge.csv', 'data/sample_eval.csv'

OUT_ROOT = '/content/results'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/kore_results'
OUT_DIR = OUT_ROOT + '/' + LABEL_NAME

py = sys.executable
common = ' --knowledge ' + KNOWLEDGE + ' --questions ' + QUESTIONS + ' --out ' + OUT_DIR + ' --splits ' + SPLITS + \
         ' --max-cands %d --max-subset %d' % (MAX_CANDS, MAX_SUBSET)
sh(py + ' -m kore label' + common + ' --dry-run --sec-per-call 0.5')   # 호출당 초는 GPU 점검 셀에서 잰 값으로 바꾼다
# 아래 줄이 실제 계산이다. 연결이 끊기면 이 셀을 다시 실행하면 이어서 돈다.
sh(py + ' -m kore label' + common + ' --generator-config ' + CONFIG, stream=True)
sh(py + ' -m kore label-report --dir ' + OUT_DIR + ' --k 2 --csv ' + OUT_DIR + '/per_question.csv')
print('위 숫자는 예비 신호다. 구간이 넓으면 결론을 내지 않고, 사전 등록(docs/PROTOCOL.md 2절) 없이 이 숫자로 주장하지 않는다.')

## 5. 블라인드 채점표

In [ ]:
# 5. 블라인드 채점표 만들기
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

USE_DRIVE = True
RUN_NAME = 'dev_run_01'
KNOWLEDGE = 'data/knowledge.csv'
QUESTIONS = 'data/evaluation.csv'
N_ITEMS = 30

os.chdir(WORKDIR)
if not os.path.exists(KNOWLEDGE) or not os.path.exists(QUESTIONS):
    KNOWLEDGE, QUESTIONS = 'data/sample_knowledge.csv', 'data/sample_eval.csv'
OUT_ROOT = '/content/drive/MyDrive/kore_results' if USE_DRIVE else '/content/results'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

RUN_DIR = OUT_ROOT + '/' + RUN_NAME
sh(sys.executable + ' -m kore blind-export --knowledge ' + KNOWLEDGE + ' --questions ' + QUESTIONS +
   ' --run-dir ' + RUN_DIR + ' --conditions vanilla_rag kore --n %d --out-dir ' % N_ITEMS + RUN_DIR + '/blind')
print('채점표:', RUN_DIR + '/blind/blind_sheet.csv')
print('채점자 두 명이 각자 복사본의 score 열만 채운다(0 틀림, 1 일부 맞음, 2 정답). blind_key.csv는 채점이 끝나기 전에는 열지 않는다.')
print('채점이 끝나면: python -m kore blind-score --sheet 채점표1.csv 채점표2.csv --key blind_key.csv')